# Quantum vs Classical - DEBUG VERSION with Progress Tracking

**REDUCED BATCH SIZE FOR FASTER ITERATIONS**

In [1]:
# Hardware-optimized configuration
import os
os.environ['OMP_NUM_THREADS'] = '16'
os.environ['MKL_NUM_THREADS'] = '16'

# Paths
DATA_DIR = "/media/priyanshu/SD/othercode/data"
SAVE_DIR = "./debug_comparison_results"

# Data parameters
MAX_DRUGS = 2000
SEED = 42

# Model parameters
NUM_QUBITS = 6
NUM_QLAYERS = 3
HIDDEN_DIM = 128

# Training parameters - REDUCED BATCH SIZE FOR VISIBLE PROGRESS
EPOCHS = 100
BATCH_SIZE = 256              # REDUCED from 512 (2x faster iterations)
LEARNING_RATE_QUANTUM = 0.005
LEARNING_RATE_CLASSICAL = 0.0005
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 15

# Hardware settings
DEVICE = 'cuda'
QUANTUM_DEVICE = 'lightning.qubit'
NUM_WORKERS = 8
PIN_MEMORY = True
VERBOSE = 2  # MAXIMUM VERBOSITY FOR DEBUGGING

print(f"DEBUG Configuration:")
print(f"  Batch size: {BATCH_SIZE} (reduced for faster feedback)")
print(f"  Verbosity: {VERBOSE} (will show every batch)")

DEBUG Configuration:
  Batch size: 256 (reduced for faster feedback)
  Verbosity: 2 (will show every batch)


In [2]:
import importlib
import drug_patient_qgnn.data_processing
import drug_patient_qgnn

importlib.reload(drug_patient_qgnn.data_processing)
importlib.reload(drug_patient_qgnn)

%load_ext autoreload
%autoreload 2

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader
import json
from tqdm import tqdm  # Progress bars!

from drug_patient_qgnn import (
    DrugPatientDataProcessor,
    QuantumDrugPatientGNN,
    set_seed,
    print_model_summary,
    print_device_info
)

set_seed(SEED)
os.makedirs(SAVE_DIR, exist_ok=True)

print_device_info()


Device Information
CUDA Available      : True
CUDA Devices        : 1
CUDA Device Name    : NVIDIA GeForce RTX 3080
MPS Available       : False



In [3]:
print(f"Loading data from {DATA_DIR}...\n")
start_time = datetime.now()

processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)

stats = processor.get_statistics()
print(f"\nData loaded in {(datetime.now() - start_time).total_seconds():.1f}s")
print("\nDataset Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key:25s}: {value:.4f}")
    else:
        print(f"  {key:25s}: {value}")

Loading data from /media/priyanshu/SD/othercode/data...

Searching for data in: /media/priyanshu/SD/othercode/data
Found 2000 protein descriptor files


Loading PDB Data: 100%|██████████| 2000/2000 [00:08<00:00, 236.34it/s]


Generating negative samples (target: 14864)...


Generating Negatives: 100%|██████████| 14864/14864 [00:00<00:00, 308869.80it/s]


Loaded 1711 proteins, 14864 drugs
Interactions: 14864 positive, 14864 negative (Total: 29728)

Data loaded in 106.8s

Dataset Statistics:
  num_ligands              : 7467
  num_pockets              : 1644
  num_interactions         : 29728
  num_drugs                : 7467
  num_patients             : 1644
  positive_rate            : 0.5000
  negative_rate            : 0.5000
  ligand_feature_dim       : 11
  drug_feature_dim         : 11
  pocket_feature_dim       : 19
  patient_feature_dim      : 19


In [ ]:
# Get graph data
graph = processor.graph
drug_features = graph.get_drug_features_matrix()
patient_features = graph.get_patient_features_matrix()
edge_index, edge_features = graph.get_edge_index()
labels = graph.get_edge_labels()

# Create interaction dataset
interaction_data = []
for idx in range(edge_index.shape[1]):
    drug_idx = int(edge_index[0, idx])
    patient_idx = int(edge_index[1, idx])
    
    interaction_data.append({
        'drug_features': drug_features[drug_idx].tolist(),
        'patient_features': patient_features[patient_idx].tolist(),

processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)
        'label': float(labels[idx])
    })


processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)
df_pandas = pd.DataFrame(interaction_data)

# Stratified split
train_pd, val_pd = train_test_split(
    df_pandas,
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=df_pandas['label']
)

print(f"\nTraining samples:   {len(train_pd):,}")
print(f"Validation samples: {len(val_pd):,}")
print(f"Batches per epoch:  {len(train_pd) // BATCH_SIZE}")
print(f"Estimated time/epoch: {(len(train_pd) // BATCH_SIZE) * 2 / 60:.1f} minutes")

# PyTorch Dataset
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.drug_features = np.stack(df['drug_features'].values)
        self.patient_features = np.stack(df['patient_features'].values)
        self.labels = df['label'].values.astype(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.drug_features[idx], dtype=torch.float32),
            torch.tensor(self.patient_features[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32),
        )

train_dataset = InteractionDataset(train_pd)
val_dataset = InteractionDataset(val_pd)

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True
)

print(f"\nDataLoaders created with {NUM_WORKERS} workers")


Training samples:   23,782
Validation samples: 5,946
Batches per epoch:  92
Estimated time/epoch: 3.1 minutes

DataLoaders created with 8 workers


In [ ]:
def train_epoch(model, optimizer, criterion, loader, device, show_progress=True):
    """Train one epoch WITH PROGRESS BAR."""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    # PROGRESS BAR FOR DEBUGGING
    pbar = tqdm(loader, desc='Training', disable=not show_progress)
    
    for batch_idx, (drug_features, patient_features, labels) in enumerate(pbar):
        batch_start = datetime.now()
        
        drug_features = drug_features.to(device, non_blocking=True)
        patient_features = patient_features.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        outputs = model(drug_features, patient_features).squeeze(-1)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Update progress bar with batch time
        batch_time = (datetime.now() - batch_start).total_seconds()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'batch_time': f'{batch_time:.1f}s'})
        

    processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
    counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)
        # Print every 10 batches for debugging
    if VERBOSE >= 2 and (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)}: loss={loss.item():.4f}, time={batch_time:.1f}s")
    
    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, (np.array(all_preds) >= 0.5).astype(int))
    
    return {'loss': avg_loss, 'accuracy': accuracy}

def evaluate(model, criterion, loader, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for drug_features, patient_features, labels in tqdm(loader, desc

processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)='Validation', leave=False):
            drug_features = drug_features.to(device, non_blocking=True)
            patient_features = patient_features.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(drug_features, patient_features).squeeze(-1)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(all_labels)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds >= 0.5).astype(int)
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy_score(all_labels, all_preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds_binary, zero_division=0),
        'recall': recall_score(all_labels, all_preds_binary, zero_division=0),
        'f1': f1_score(all_labels, all_preds_binary, zero_division=0)
    }

def train_model(model, train_loader, val_loader, learning_rate, model_name, device):
    """Train with detailed progress tracking."""
    print(f"\n{'='*70}")
    print(f"Training {model_name.upper()} Model")
    print(f"{'='*70}")
    print(f"Learning Rate: {learning_rate}")
    print(f"Device: {device}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Batches per epoch: {len(train_loader)}")
    print(f"Estimated time/epoch: ~{len(train_loader) * 2 / 60:.1f} min\n")
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )
    criterion = torch.nn.BCEWithLogitsLoss()

processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_auc': [],
        'val_precision': [], 'val_recall': [], 'val_f1': [],
        'learning_rates': []
    }
    
    best_val_auc = 0.0
    patience_counter = 0
    start_time = datetime.now()
    
    for epoch in range(EPOCHS):
        epoch_start = datetime.now()
        print(f"\n{'='*70}")
        print(f"EPOCH {epoch+1}/{EPOCHS} - Started at {epoch_start.strftime('%H:%M:%S')}")
        print(f"{'='*70}")
        
        train_metrics = train_epoch(model, optimizer, criterion, train_l

processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)oader, device)
        val_metrics = evaluate(model, criterion, val_loader, device)
        
        # Update scheduler
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_metrics['auc'])
        current_lr = optimizer.param_groups[0]['lr']
        
        if current_lr != old_lr:
            print(f"  Learning rate reduced: {old_lr:.6f} → {current_lr:.6f}")
        
        # Save metrics
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_auc'].append(val_metrics['auc'])
        history['val_precision'].append(val_metrics['precision'])
        history['val_recall'].append(val_metrics['recall'])
        history['val_f1'].append(val_metrics['f1'])
        history['learning_rates'].append(current_lr)
        
        epoch_time = (datetime.now() - epoch_start).total_seconds()
        
        # Print results
        print(f"\n{'-'*70}")
        print(f"Epoch {epoch+1} Results ({epoch_time:.1f}s):")
        print(f"  Train: loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}")
        print(f"  Val:   loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
              f"auc={val_metrics['auc']:.4f}, f1={val_metrics['f1']:.4f}")
        print(f"  Best AUC so far: {best_val_auc:.4f}")
        print(f"{'-'*70}")
        
        # Save best model
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_val_auc,
                'history': history
            }, os.path.join(SAVE_DIR, f"{model_name}_best.pt"))
            
            print(f"✓ New best AUC: {best_val_auc:.4f} (saved)")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
        
        # Auto-save every epoch for debugging
        with open(os.path.join(SAVE_DIR, f"{model_name}_history.json"), 'w') as f:
            json.dump(history, f, indent=2)
        
        # Early stopping
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    total_time = (datetime.now() - start_time).total_seconds()
    print(f"\n{model_name.upper()} Training Complete!")
    print(f"  Total time: {total_time/3600:.2f} hours")
    print(f"  Best AUC: {best_val_auc:.4f}")
    
    return history, best_val_auc



✓ Training functions defined with progress bars


In [ ]:
drug_dim = len(drug_features[0])
patient_dim = len(patient_features[0])

print("Creating Quantum Model...")
quantum_model = QuantumDrugPatientGNN(
    drug_dim=drug_dim,
    patient_dim=patient_dim,
    num_qubits=NUM_QUBITS,
    num_qlayers=NUM_QLAYERS,

processor = DrugPatientDataProcessor(data_dir=DATA_DIR, seed=SEED)
counts = processor.load_real_data(data_dir=DATA_DIR, max_samples=MAX_DRUGS)
    hidden_dim=HIDDEN_DIM,
    use_quantum=True,
    device_name=QUANTUM_DEVICE
)

print_model_summary(quantum_model, drug_dim, patient_dim)

print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

quantum_history, quantum_best_auc = train_model(
    quantum_model,
    train_loader,
    val_loader,
    LEARNING_RATE_QUANTUM,
    "quantum",
    DEVICE
)

print(f"\nQuantum training finished at: {datetime.now().strftime('%H:%M:%S')}")Host ist-jump
    HostName ssh.ist.psu.edu
    User pkd5228@psu.edu
Host nittany-ai-compute
    HostName nittanyaicompute.ist.psu.edu
    User pkd5228@psu.edu
    ProxyJump ist-jump


Creating Quantum Model...

Model Summary
ligand_dim          : 11
pocket_dim          : 19
num_qubits          : 6
num_qlayers         : 3
use_quantum         : True
num_parameters      : 6225
drug_dim            : 11
patient_dim         : 19
Total parameters    : 6,225
Trainable params    : 6,225
Input (drug)        : (11,)
Input (patient)     : (19,)
Output              : (1,) [probability]


Started at: 2025-12-01 01:12:28


Training QUANTUM Model
Learning Rate: 0.005
Device: cuda
Parameters: 6,225
Batches per epoch: 93
Estimated time/epoch: ~3.1 min


EPOCH 1/100 - Started at 01:12:29


Training:  11%|█         | 10/93 [06:46<55:59, 40.48s/it, loss=0.6912, batch_time=40.4s] 

  Batch 10/93: loss=0.6912, time=40.4s


Training:  22%|██▏       | 20/93 [13:31<49:24, 40.61s/it, loss=0.6830, batch_time=40.9s]

  Batch 20/93: loss=0.6830, time=40.9s


...

Training:  97%|█████████▋| 90/93 [1:00:40<02:01, 40.35s/it, loss=0.5977, batch_time=40.2s]

  Batch 90/93: loss=0.5977, time=40.2s


Training: 100%|██████████| 93/93 [1:02:38<00:00, 40.41s/it, loss=0.5691, batch_time=36.6s]



----------------------------------------------------------------------
Epoch 7 Results (3833.1s):
  Train: loss=0.5932, acc=0.6655
  Val:   loss=0.5999, acc=0.6643, auc=0.7309, f1=0.6657
  Best AUC so far: 0.7327
----------------------------------------------------------------------
Patience: 1/15

EPOCH 8/100 - Started at 08:40:48


Training:   3%|▎         | 3/93 [02:01<1:00:39, 40.44s/it, loss=0.5870, batch_time=40.2s]